In [1]:
pip install torch torchvision pillow pillow-heif tqdm sklearn -q

Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://github.com/scikit-learn/sklearn-pypi-packag

In [13]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [4]:
def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="Training", leave=False)

    for x, y in pbar:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = loss_fn(logits, y)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += x.size(0)

        avg_loss = total_loss / total
        avg_acc = correct / total
        pbar.set_postfix(loss=f"{avg_loss:.4f}", acc=f"{avg_acc:.4f}")

    return total_loss / total, correct / total

In [5]:
@torch.no_grad() # 함수 전체에 torch.no_grad 적용
def run_eval(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = loss_fn(logits, y)

        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += x.size(0)

    avg_loss = total_loss / max(total, 1)
    acc = correct / max(total, 1)
    return avg_loss, acc

In [6]:
weights = ViT_B_16_Weights.DEFAULT
model = vit_b_16(weights=weights)

train_dir = r"C:\temp_test\260224-ViT\data\train"

In [7]:
# 클래스 수 확인
dummy_ds = ImageFolder(train_dir)
num_classes = len(dummy_ds.classes)

print(f"총 클래스 수 {num_classes}개")

for class_name, class_idx in dummy_ds.class_to_idx.items():
    count = sum(1 for target in dummy_ds.targets if target == class_idx)
    print(f"{class_name}: {count}개")

총 클래스 수 4개
0-building-in: 17개
1-building-out: 32개
2-road: 44개
3-nature: 40개


In [8]:
imgnt_mean = (0.485, 0.456, 0.406)
imgnt_std  = (0.229, 0.224, 0.225)

tf = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BILINEAR), # 256으로 resize해준 다음에 224로 center crop 하는게 ViT pretraining 조건이라 똑같이 맞춰줬음
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(imgnt_mean, imgnt_std),
])

train_data = ImageFolder(train_dir, transform=tf)

train_loader = DataLoader(
    train_data,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=(device == "cuda")
)

In [9]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
loss_fn = nn.CrossEntropyLoss()

In [10]:
def build_model(num_classes: int):
    m = vit_b_16(weights=weights)
    in_features = m.heads.head.in_features
    m.heads.head = nn.Linear(in_features, num_classes)
    return m.to(device)

In [11]:
# K-Fold 설정
k = 5
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

epochs = 10
batch_size = 8
fold_results = []

In [15]:
y = np.array(train_data.targets)

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(y)), y), start=1):
    print(f"\n===== Fold {fold}/{k} =====")

    model = build_model(num_classes)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
    loss_fn = nn.CrossEntropyLoss()

    train_subset = Subset(train_data, train_idx)
    val_subset   = Subset(train_data, val_idx)

    train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=(device == "cuda")
    )
    val_loader = DataLoader(
        val_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=(device == "cuda")
    )

    best_val_acc = 0.0
    best_val_loss = float("inf")

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
        val_loss, val_acc = run_eval(model, val_loader, loss_fn, device)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss

        print(
            f"[Fold {fold} | Epoch {epoch:02d}] "
            f"train loss={train_loss:.4f}, acc={train_acc:.4f} | "
            f"val loss={val_loss:.4f}, acc={val_acc:.4f}"
        )

    fold_results.append((best_val_loss, best_val_acc))
    print(f"Fold {fold} Best: val loss={best_val_loss:.4f}, val acc={best_val_acc:.4f}")

# 최종 평균
avg_loss = sum(r[0] for r in fold_results) / len(fold_results)
avg_acc  = sum(r[1] for r in fold_results) / len(fold_results)
print(f"\n===== CV Result ({k}-fold) =====")
print(f"Avg best val loss={avg_loss:.4f}, Avg best val acc={avg_acc:.4f}")


===== Fold 1/5 =====


[Fold 1 | Epoch 01] train loss=1.2852, acc=0.4434 | val loss=0.7802, acc=0.7037


[Fold 1 | Epoch 02] train loss=1.1893, acc=0.5849 | val loss=1.1159, acc=0.5556


[Fold 1 | Epoch 03] train loss=0.6831, acc=0.7830 | val loss=1.2156, acc=0.5926


[Fold 1 | Epoch 04] train loss=0.4207, acc=0.8585 | val loss=0.3319, acc=0.8519


...

[Fold 5 | Epoch 06] train loss=0.2085, acc=0.9159 | val loss=0.8472, acc=0.7308


[Fold 5 | Epoch 07] train loss=0.3243, acc=0.8785 | val loss=0.5288, acc=0.8077


[Fold 5 | Epoch 08] train loss=0.1150, acc=0.9626 | val loss=0.5023, acc=0.8462


[Fold 5 | Epoch 09] train loss=0.0516, acc=0.9907 | val loss=2.0152, acc=0.6923


[Fold 5 | Epoch 10] train loss=0.0308, acc=0.9907 | val loss=1.9825, acc=0.5769
Fold 5 Best: val loss=0.5023, val acc=0.8462

===== CV Result (5-fold) =====
Avg best val loss=0.6644, Avg best val acc=0.7510
